# Phase 2: Open-Domain Societal Generalization (SWA-MPPI)

This notebook executes the SWA-MPPI framework on open-ended generative tasks, utilizing **Union Sampling** and the **Dynamic Variance Trigger** to achieve latency reductions and robust cultural consensus. 
Target Datasets: **GlobalOpinionQA**, **BLEnD**, and **SPB-2602**.

Data is loaded from Kaggle input: `/kaggle/input/datasets/minh2duy/swa-mppi-phase2-data`.


In [1]:
!pip install unsloth==2026.3.4 torch==2.10.0 torchvision==0.25.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.2/447.2 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.7/915.7 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 114.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 130.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 134.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 60.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [2]:
"""
run_phase2_open_domain.py — NeurIPS-Grade SWA-MPPI Phase 2 Evaluation
=====================================================================

Dynamic Social Consensus for Cross-Cultural Value Negotiation
via Implicit Pre-Logit Control

Phase 2: Open-Domain Societal Generalization
  - GlobalOpinionQA  → JSD between LLM response distribution and human survey data
  - BLEnD MCQ        → Cross-linguistic cultural accuracy on 16 country benchmarks

Architecture:
  1. Load diverse demographic personas from SPB-2602  (N agents)
  2. For each question, run autoregressive SWA-MPPI generation with
     Top-K Union Sampling (pruned search space) and variance trigger
  3. Evaluate response distributions against ground-truth human surveys
  4. Produce publication-quality figures (PDF/PNG) for NeurIPS submission

--------------------------------------------------------------------------------
KAGGLE SETUP (copy script + data lên Kaggle, chạy Notebook/Script):
--------------------------------------------------------------------------------
  1. Tạo Notebook mới → Settings → Accelerator: GPU (T4 P100 hoặc better).
  2. Add Data: thêm dataset chứa Phase 2 (llm_global_opinions_train.csv,
     BLEnD_multiple-choice-questions_test.csv, SPB-2602_train.csv).
  3. Trong Notebook, chạy 1 ô cài đặt:
     !pip install unsloth==2026.3.4 torch==2.10.0 torchvision==0.25.0
  4. Copy toàn bộ nội dung file này vào 1 cell hoặc upload file .py rồi:
     exec(open("run_phase2_open_domain.py").read())   hoặc   %run run_phase2_open_domain.py
  5. Kết quả lưu tại /kaggle/working/SWA_MPPI_Phase2/results/
     (CSV, PDF/PNG figures, summaries). Download từ Output của Notebook.
  Env (optional): SWA_MPPI_DATA=/kaggle/input/<your-dataset-slug> nếu path khác.
  Chạy nhanh (~20–40 phút): set env SWA_MPPI_FAST=1 hoặc thêm --fast khi gọi script.
"""

import os
import ast
import json
import argparse
import time
import gc
import pickle
import warnings
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass, field
from collections import defaultdict
from scipy.spatial.distance import jensenshannon

try:
    from unsloth import FastLanguageModel
except ImportError:
    print("[ERROR] The 'unsloth' package is not installed or not compatible with your environment.")
    print("Please install 'unsloth' and all required dependencies manually, ensuring CUDA and PyTorch versions are compatible.")
    print("Example: pip install unsloth==2026.3.4 torch==2.10.0 torchvision==0.25.0")
    raise SystemExit(1)


def _is_kaggle() -> bool:
    """True when running inside Kaggle kernel (Notebook or Script)."""
    return (
        os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None
        or (Path("/kaggle").exists() and Path("/kaggle/working").exists())
    )


def _resolve_kaggle_data_dir(configured_data_dir: str) -> Path:
    """
    Resolve data directory on Kaggle: use configured path if it exists and has
    required files; else search /kaggle/input for a directory containing them.
    """
    path = Path(configured_data_dir)
    required_files = [
        "llm_global_opinions_train.csv",
        "SPB-2602_train.csv",
        "BLEnD_multiple-choice-questions_test.csv",
    ]
    def has_required(p: Path) -> bool:
        return p.is_dir() and all((p / f).exists() for f in required_files)

    if has_required(path):
        return path
    env_data = os.environ.get("SWA_MPPI_DATA")
    if env_data and has_required(Path(env_data)):
        return Path(env_data)
    base = Path("/kaggle/input")
    if base.exists():
        for d in sorted(base.iterdir()):
            if d.is_dir() and has_required(d):
                return d
            # One level deeper: e.g. /kaggle/input/owner-dataset-name/subfolder
            for sub in d.iterdir() if d.is_dir() else []:
                if sub.is_dir() and has_required(sub):
                    return sub
    return path  # Return original so FileNotFoundError message is clear


# ============================================================================
# 1. CONFIGURATION & RESEARCH PLOTTING SETUP
# ============================================================================
warnings.filterwarnings("ignore", category=UserWarning, module="torch")
warnings.filterwarnings("ignore", category=FutureWarning)

torch.set_float32_matmul_precision("high")

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
})


@dataclass
class Phase2Config:
    """All hyperparameters for Phase 2 open-domain evaluation."""
    model_name: str = "unsloth/Meta-Llama-3.1-70B-Instruct-bnb-4bit"
    max_seq_length: int = 4096
    load_in_4bit: bool = True

    # Data paths (Kaggle dataset mount)
    data_dir: str = "/kaggle/input/datasets/minh2duy/swa-mppi-phase2-data"
    output_dir: str = "/kaggle/working/SWA_MPPI_Phase2/results"

    # SWA-MPPI Core Hyperparameters (matches paper Eq 2-4)
    lambda_coop: float = 0.7       # Cooperation parameter
    alpha_kl: float = 0.05         # KL-divergence penalty
    K_union: int = 50              # Top-K union sampling (paper spec)
    tau_conflict: float = 0.005    # Variance trigger threshold
    mppi_trajectories: int = 64    # MPPI perturbation samples
    mppi_noise_std: float = 0.5    # Noise std on pruned logit subspace
    mppi_temp: float = 0.5         # MPPI inverse temperature

    # Generation
    max_new_tokens: int = 150

    # Evaluation (use smaller defaults when SWA_MPPI_FAST=1 for quicker Kaggle runs)
    num_eval_globalopinion: int = 100   # Questions from GlobalOpinionQA
    num_eval_blend: int = 200           # Questions from BLEnD MCQ
    num_agents: int = 5                 # Demographic agents per evaluation

    # BLEnD target countries (matching the 16 available short-answer CSVs)
    blend_countries: List[str] = field(default_factory=lambda: [
        "UK", "US", "China", "South_Korea", "Mexico", "Ethiopia",
        "Greece", "Indonesia", "Iran", "Spain", "Algeria",
        "Azerbaijan", "Assam", "North_Korea", "Northern_Nigeria", "West_Java",
    ])


def parse_args() -> Phase2Config:
    parser = argparse.ArgumentParser(description="SWA-MPPI Phase 2: Open-Domain Evaluation")
    parser.add_argument("--model_name", type=str, default=Phase2Config.model_name)
    parser.add_argument("--data_dir", type=str, default=Phase2Config.data_dir)
    parser.add_argument("--output_dir", type=str, default=Phase2Config.output_dir)
    parser.add_argument("--num_eval_globalopinion", type=int, default=Phase2Config.num_eval_globalopinion)
    parser.add_argument("--num_eval_blend", type=int, default=Phase2Config.num_eval_blend)
    parser.add_argument("--num_agents", type=int, default=Phase2Config.num_agents)
    parser.add_argument("--tau", type=float, default=Phase2Config.tau_conflict)
    parser.add_argument("--lambda_c", type=float, default=Phase2Config.lambda_coop)
    parser.add_argument("--K_union", type=int, default=Phase2Config.K_union)
    parser.add_argument("--max_new_tokens", type=int, default=Phase2Config.max_new_tokens)
    parser.add_argument("--fast", action="store_true", help="Fewer questions & shorter gen for quick Kaggle run (~20–40 min)")
    args, _ = parser.parse_known_args()

    # Fast mode: fewer samples + shorter generation (full forward each step = very slow on Kaggle)
    # Default ON when on Kaggle unless SWA_MPPI_FAST=0; or use --fast / SWA_MPPI_FAST=1 elsewhere
    if args.fast or os.environ.get("SWA_MPPI_FAST") == "1" or (
        _is_kaggle() and os.environ.get("SWA_MPPI_FAST") != "0"
    ):
        args.num_eval_globalopinion = min(args.num_eval_globalopinion, 8)
        args.num_eval_blend = min(args.num_eval_blend, 16)
        args.max_new_tokens = min(args.max_new_tokens, 64)
        print("[CONFIG] Fast mode (Kaggle): 8 GlobalOpinion, 16 BLEnD, max_new_tokens=64 (~20–40 min)")

    return Phase2Config(
        model_name=args.model_name,
        data_dir=args.data_dir,
        output_dir=args.output_dir,
        num_eval_globalopinion=args.num_eval_globalopinion,
        num_eval_blend=args.num_eval_blend,
        num_agents=args.num_agents,
        tau_conflict=args.tau,
        lambda_coop=args.lambda_c,
        K_union=args.K_union,
        max_new_tokens=args.max_new_tokens,
    )


# ============================================================================
# 2. DATA PIPELINE
# ============================================================================
def _safe_parse_dict(s: str) -> dict:
    """Parse a stringified defaultdict or dict safely."""
    s = s.strip()
    if s.startswith("defaultdict"):
        # Extract the dict portion from defaultdict(<class 'list'>, {…})
        start = s.index("{")
        end = s.rindex("}") + 1
        return ast.literal_eval(s[start:end])
    return ast.literal_eval(s)


def load_globalopinion_data(data_dir: Path, n: int) -> pd.DataFrame:
    """Load and preprocess GlobalOpinionQA questions."""
    path = data_dir / "llm_global_opinions_train.csv"
    df = pd.read_csv(path)
    df = df.head(n).copy()
    # Parse options from string to list
    df["options_list"] = df["options"].apply(ast.literal_eval)
    # Parse selections (country → distribution)
    df["selections_dict"] = df["selections"].apply(_safe_parse_dict)
    print(f"[DATA] Loaded {len(df)} GlobalOpinionQA questions")
    return df


def load_blend_mcq_data(data_dir: Path, n: int) -> pd.DataFrame:
    """Load BLEnD multiple-choice questions."""
    path = data_dir / "BLEnD_multiple-choice-questions_test.csv"
    df = pd.read_csv(path)
    # Sample stratified by country for diversity
    sampled = df.groupby("country", group_keys=False).apply(
        lambda x: x.sample(n=min(len(x), max(1, n // df["country"].nunique())),
                           random_state=42)
    )
    sampled = sampled.head(n).reset_index(drop=True)
    print(f"[DATA] Loaded {len(sampled)} BLEnD MCQ questions ({sampled['country'].nunique()} countries)")
    return sampled


def load_personas(data_dir: Path, n_agents: int) -> List[str]:
    """Load diverse demographic personas from SPB-2602."""
    path = data_dir / "SPB-2602_train.csv"
    df = pd.read_csv(path)
    agents = df["persona_text"].sample(n=n_agents, random_state=42).tolist()
    agents = [p.replace("<character>", "").strip() for p in agents]
    print(f"[DATA] Loaded {len(agents)} demographic personas from SPB-2602")
    return agents


# ============================================================================
# 3. MULTI-AGENT KV-CACHE MANAGER
# ============================================================================
class MultiAgentKVManager:
    """Manages KV-caches for N agents + 1 base model for O(1) step generation."""
    def __init__(self, num_agents: int):
        self.caches = {i: None for i in range(num_agents)}
        self.base_cache = None

    def update_agent(self, agent_idx: int, kv):
        self.caches[agent_idx] = kv

    def update_base(self, kv):
        self.base_cache = kv

    def get_agent(self, agent_idx: int):
        return self.caches[agent_idx]

    def get_base(self):
        return self.base_cache

    def clear(self):
        self.caches = {i: None for i in self.caches}
        self.base_cache = None


# ============================================================================
# 4. CORE ENGINE: SWA-MPPI PHASE 2 (Top-K Union Sampling)
# ============================================================================
class Phase2SWAEngine:
    """
    SWA-MPPI Phase 2 Engine for open-ended text generation.

    Key difference from Phase 1 (decision-focused 1D):
      - Operates on the PRUNED Top-K union subspace (~50 tokens)
      - Uses variance trigger to skip MPPI on syntactic tokens (~70-85% savings)
      - Multi-agent KV-cache sharing for prefix caching
    """

    def __init__(
        self,
        cfg: Phase2Config,
        model,
        tokenizer,
        personas: List[str],
        name: str = "SWA_MPPI",
        enable_mppi: bool = True,
        enable_variance_trigger: bool = True,
    ):
        self.cfg = cfg
        self.model = model
        self.tokenizer = tokenizer
        self.personas = personas
        self.N = len(personas)
        self.device = next(model.parameters()).device
        self.name = name
        self.enable_mppi = bool(enable_mppi)
        self.enable_variance_trigger = bool(enable_variance_trigger)

        # Pre-build system prompt token sequences
        self.agent_prompts = []
        for p in self.personas:
            p_fmt = (
                f"<|start_header_id|>system<|end_header_id|>\n\n"
                f"You are {p}.<|eot_id|>"
                f"<|start_header_id|>user<|end_header_id|>\n\n"
            )
            ids = self.tokenizer(p_fmt, return_tensors="pt").input_ids.to(self.device)
            self.agent_prompts.append(ids)

        base_fmt = (
            "<|start_header_id|>system<|end_header_id|>\n\n"
            "You are a helpful assistant.<|eot_id|>"
            "<|start_header_id|>user<|end_header_id|>\n\n"
        )
        self.base_prompt = self.tokenizer(base_fmt, return_tensors="pt").input_ids.to(self.device)

        self.pad_id = (
            tokenizer.pad_token_id
            if tokenizer.pad_token_id is not None
            else tokenizer.eos_token_id
        )
        print(f"[ENGINE] Phase2SWAEngine initialized: {self.N} agents, K_union={cfg.K_union}, name={self.name}")

    @torch.no_grad()
    def _run_aisp_control(
        self, base_logits: torch.Tensor, top_k_indices: torch.Tensor,
        full_agent_ids_list: List[torch.Tensor],
    ) -> int:
        """
        Adaptive Importance Sampling on Pre-logits (Eq 6 from paper).
        Uses full forward (no KV cache) for Unsloth compatibility.
        Returns the selected token index in the full vocabulary.
        """
        K = top_k_indices.shape[0]
        base_subset = base_logits[top_k_indices]   # (K,)
        base_probs = F.softmax(base_subset, dim=-1)

        # Full forward per agent (Unsloth paged cache is fixed-size; avoid incremental)
        agent_subsets = []
        for i in range(self.N):
            out = self.model(
                full_agent_ids_list[i],
                use_cache=False,
                return_dict=True,
            )
            ag_logits = out.logits[0, -1, :]
            agent_subsets.append(ag_logits[top_k_indices])

        agent_subsets = torch.stack(agent_subsets)  # (N, K)

        # Contrastive rewards: r_i = log P(tok|Agent_i) - log P(tok|Base)
        r_i = (F.log_softmax(agent_subsets, dim=-1)
               - F.log_softmax(base_subset.unsqueeze(0), dim=-1))  # (N, K)

        # MPPI trajectories
        epsilon = torch.randn(
            self.cfg.mppi_trajectories, K, device=self.device
        ) * self.cfg.mppi_noise_std
        z_perturbed = base_subset.unsqueeze(0) + epsilon  # (Tr, K)
        P_pert = F.softmax(z_perturbed, dim=-1)           # (Tr, K)

        # Compute SWA utility per trajectory (vectorized over agents)
        U_total = torch.zeros(self.cfg.mppi_trajectories, device=self.device)

        for tr in range(self.cfg.mppi_trajectories):
            P_tr = P_pert[tr]
            kl = F.kl_div(P_tr.log(), base_probs, reduction="sum")

            for i in range(self.N):
                r_exp = torch.sum(P_tr * r_i[i])
                r_social = (torch.sum(P_tr * r_i).item() - r_exp.item()) / max(1, self.N - 1)
                u_i = ((1 - self.cfg.lambda_coop) * r_exp
                       + self.cfg.lambda_coop * r_social
                       - self.cfg.alpha_kl * kl)
                U_total[tr] += u_i

        # MPPI-weighted shift
        weights = F.softmax(U_total / self.cfg.mppi_temp, dim=0)  # (Tr,)
        delta_z = torch.sum(weights.unsqueeze(1) * epsilon, dim=0)  # (K,)

        best_idx = torch.argmax(base_subset + delta_z)
        return top_k_indices[best_idx]

    @torch.no_grad()
    def generate(self, user_query: str) -> Tuple[str, float, Dict]:
        """
        Autoregressive generation with SWA-MPPI on pruned Top-K subspace.
        Uses full forward each step (no KV cache in loop) for Unsloth compatibility:
        Unsloth's paged KV cache is fixed-size at prefill, so incremental decode causes IndexError.
        """
        query_ids = self.tokenizer(
            f"{user_query}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n",
            return_tensors="pt", add_special_tokens=False,
        ).input_ids.to(self.device)

        # Full sequence = prefix + query + generated; we grow generated_tokens each step
        generated_tokens = []
        triggers = 0
        steps = 0
        variances = []

        for _ in range(self.cfg.max_new_tokens):
            # Build full sequence for base: base_prompt + query_ids + generated_so_far
            if generated_tokens:
                gen_tensor = torch.tensor(
                    [generated_tokens], device=self.device, dtype=query_ids.dtype
                )
                full_base_ids = torch.cat([self.base_prompt, query_ids, gen_tensor], dim=1)
            else:
                full_base_ids = torch.cat([self.base_prompt, query_ids], dim=1)

            out_base = self.model(full_base_ids, use_cache=False, return_dict=True)
            base_logits = out_base.logits[0, -1, :]

            # Top-K union (Section 6.2 of paper)
            _, top_k_indices = torch.topk(base_logits, self.cfg.K_union)

            # Variance trigger: normalized entropy as proxy for inter-agent conflict
            p_sub = F.softmax(base_logits[top_k_indices], dim=0)
            entropy = -torch.sum(p_sub * torch.log(p_sub + 1e-10)).item()
            var_proxy = entropy / np.log(self.cfg.K_union)
            variances.append(var_proxy)

            do_mppi = False
            if self.enable_mppi:
                if self.enable_variance_trigger:
                    do_mppi = (var_proxy > self.cfg.tau_conflict and steps > 2)
                else:
                    do_mppi = (steps > 2)

            if do_mppi:
                triggers += 1
                # Build full sequences for each agent (same prefix + query + generated_so_far)
                full_agent_list = []
                for i in range(self.N):
                    if generated_tokens:
                        gen_t = torch.tensor(
                            [generated_tokens], device=self.device, dtype=query_ids.dtype
                        )
                        full_agent_list.append(
                            torch.cat([self.agent_prompts[i], query_ids, gen_t], dim=1)
                        )
                    else:
                        full_agent_list.append(
                            torch.cat([self.agent_prompts[i], query_ids], dim=1)
                        )
                next_id = self._run_aisp_control(
                    base_logits, top_k_indices, full_agent_list,
                )
            else:
                next_id = top_k_indices[0]

            next_id = next_id.unsqueeze(0).unsqueeze(0)
            generated_tokens.append(next_id.item())
            steps += 1

            if next_id.item() in [self.tokenizer.eos_token_id, 128009]:
                break

        response = self.tokenizer.decode(generated_tokens, skip_special_tokens=True)
        trigger_rate = (triggers / max(1, steps)) * 100
        mean_variance = float(np.mean(variances)) if variances else 0.0
        diagnostics = {
            "steps": steps,
            "triggers": triggers,
            "mean_variance": mean_variance,
        }
        return response, trigger_rate, diagnostics


# ============================================================================
# 4B. BASELINES: Standard Generation (for NeurIPS-style comparisons)
# ============================================================================
class BaseGenerationEngine:
    """
    Baseline text generation using HF/Unsloth `model.generate`.
    This avoids our manual KV-cache logic and is a strong, standard baseline.
    """
    def __init__(
        self,
        cfg: Phase2Config,
        model,
        tokenizer,
        name: str,
        do_sample: bool,
        temperature: float = 1.0,
        top_p: float = 1.0,
        top_k: int = 0,
    ):
        self.cfg = cfg
        self.model = model
        self.tokenizer = tokenizer
        self.name = name
        self.do_sample = do_sample
        self.temperature = float(temperature)
        self.top_p = float(top_p)
        self.top_k = int(top_k)
        self.device = next(model.parameters()).device

        base_fmt = (
            "<|start_header_id|>system<|end_header_id|>\n\n"
            "You are a helpful assistant.<|eot_id|>"
            "<|start_header_id|>user<|end_header_id|>\n\n"
        )
        self.base_prompt = self.tokenizer(base_fmt, return_tensors="pt").input_ids.to(self.device)

    @torch.no_grad()
    def generate(self, user_query: str) -> Tuple[str, float, Dict]:
        query_ids = self.tokenizer(
            f"{user_query}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n",
            return_tensors="pt",
            add_special_tokens=False,
        ).input_ids.to(self.device)
        input_ids = torch.cat([self.base_prompt, query_ids], dim=1)

        # Note: `model.generate` handles cache internally & efficiently.
        gen_ids = self.model.generate(
            input_ids=input_ids,
            max_new_tokens=self.cfg.max_new_tokens,
            do_sample=self.do_sample,
            temperature=self.temperature if self.do_sample else 1.0,
            top_p=self.top_p if self.do_sample else 1.0,
            top_k=self.top_k if self.do_sample else 0,
            pad_token_id=self.tokenizer.eos_token_id,
            eos_token_id=self.tokenizer.eos_token_id,
        )

        # Only decode newly generated tokens (exclude prompt)
        new_tokens = gen_ids[0, input_ids.shape[1]:].tolist()
        response = self.tokenizer.decode(new_tokens, skip_special_tokens=True)
        diagnostics = {
            "steps": int(len(new_tokens)),
            "triggers": 0,
            "mean_variance": 0.0,
        }
        return response, 0.0, diagnostics


class SelfConsistencyEngine:
    """
    SOTA-ish baseline: sample N times then aggregate.
      - GlobalOpinionQA: average option-prob vectors extracted from each sample
      - BLEnD MCQ: majority vote over extracted choices
    """
    def __init__(
        self,
        cfg: Phase2Config,
        model,
        tokenizer,
        name: str,
        n_samples: int = 5,
        temperature: float = 0.7,
        top_p: float = 0.9,
        top_k: int = 0,
    ):
        self.cfg = cfg
        self.model = model
        self.tokenizer = tokenizer
        self.name = name
        self.n_samples = int(n_samples)
        self.temperature = float(temperature)
        self.top_p = float(top_p)
        self.top_k = int(top_k)
        self.device = next(model.parameters()).device

        base_fmt = (
            "<|start_header_id|>system<|end_header_id|>\n\n"
            "You are a helpful assistant.<|eot_id|>"
            "<|start_header_id|>user<|end_header_id|>\n\n"
        )
        self.base_prompt = self.tokenizer(base_fmt, return_tensors="pt").input_ids.to(self.device)

    @torch.no_grad()
    def _sample_once(self, user_query: str) -> str:
        query_ids = self.tokenizer(
            f"{user_query}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n",
            return_tensors="pt",
            add_special_tokens=False,
        ).input_ids.to(self.device)
        input_ids = torch.cat([self.base_prompt, query_ids], dim=1)

        gen_ids = self.model.generate(
            input_ids=input_ids,
            max_new_tokens=self.cfg.max_new_tokens,
            do_sample=True,
            temperature=self.temperature,
            top_p=self.top_p,
            top_k=self.top_k,
            pad_token_id=self.tokenizer.eos_token_id,
            eos_token_id=self.tokenizer.eos_token_id,
        )
        new_tokens = gen_ids[0, input_ids.shape[1]:].tolist()
        return self.tokenizer.decode(new_tokens, skip_special_tokens=True)

    @torch.no_grad()
    def generate(self, user_query: str) -> Tuple[str, float, Dict]:
        # Return first sample as "response" (we aggregate elsewhere for metrics)
        samples = [self._sample_once(user_query) for _ in range(self.n_samples)]
        diagnostics = {
            "steps": 0,
            "triggers": 0,
            "mean_variance": 0.0,
            "n_samples": self.n_samples,
            "samples": samples,
        }
        return samples[0] if samples else "", 0.0, diagnostics


# ============================================================================
# 5. EVALUATION: GlobalOpinionQA (JSD against human survey distributions)
# ============================================================================
def _extract_option_probs_from_response(
    response: str, options: List[str],
) -> np.ndarray:
    """
    Heuristic: map generated text to a soft distribution over options.
    Assigns probability mass based on which options are mentioned/selected.
    Handles options that may be str, float (e.g. NaN), or other types from CSV.
    """
    response_lower = (response or "").lower().strip()
    scores = np.zeros(len(options))

    for i, opt in enumerate(options):
        if opt is None or (isinstance(opt, float) and np.isnan(opt)):
            continue
        opt_lower = str(opt).strip().lower()
        # Skip DK/Refused type options for matching
        if any(skip in opt_lower for skip in ["dk/", "refused", "don't know"]):
            continue
        if opt_lower in response_lower:
            scores[i] += 2.0
        # Check for partial keyword match (first 3+ chars words)
        keywords = [w for w in opt_lower.split() if len(w) > 3]
        for kw in keywords:
            if kw in response_lower:
                scores[i] += 0.5

    # If no match found, distribute uniformly
    if scores.sum() < 1e-10:
        scores = np.ones(len(options))

    # Normalize to probability distribution
    scores = np.clip(scores, 1e-10, None)
    return scores / scores.sum()


def evaluate_globalopinion(
    engine, df: pd.DataFrame, cfg: Phase2Config, method_name: str = "SWA_MPPI",
) -> Tuple[pd.DataFrame, Dict]:
    """
    Evaluate on GlobalOpinionQA: JSD between LLM response distribution and human survey.
    """
    print(f"\n[EVAL] GlobalOpinionQA — {len(df)} questions")
    results = []
    country_jsd_accum = defaultdict(list)  # country → [jsd per question]

    for idx, row in tqdm(df.iterrows(), total=len(df), desc="GlobalOpinionQA"):
        question = row["question"]
        options = row["options_list"]
        selections = row["selections_dict"]  # {country: [p1, p2, ...]}

        # Build prompt
        opts_text = "\n".join([f"  {chr(65+i)}. {o}" for i, o in enumerate(options)])
        prompt = (
            f"Question: {question}\n\n"
            f"Options:\n{opts_text}\n\n"
            f"Please select the most appropriate option and briefly explain your reasoning."
        )

        t0 = time.time()
        resp, t_rate, diag = engine.generate(prompt)
        latency = time.time() - t0

        # Extract model's implied distribution over options
        if isinstance(diag, dict) and "samples" in diag:
            # self-consistency: average over samples
            dists = [
                _extract_option_probs_from_response(s, options)
                for s in diag.get("samples", [])
            ]
            model_dist = np.mean(dists, axis=0) if len(dists) else _extract_option_probs_from_response(resp, options)
        else:
            model_dist = _extract_option_probs_from_response(resp, options)

        # Compute JSD against each country's human distribution
        per_country_jsd = {}
        for country, human_dist_raw in selections.items():
            human_dist = np.array(human_dist_raw, dtype=np.float64)
            if len(human_dist) != len(options):
                continue
            human_dist = np.clip(human_dist, 1e-10, None)
            human_dist = human_dist / human_dist.sum()
            jsd = float(jensenshannon(model_dist, human_dist))
            per_country_jsd[country] = jsd
            country_jsd_accum[country].append(jsd)

        mean_jsd = float(np.mean(list(per_country_jsd.values()))) if per_country_jsd else np.nan

        results.append({
            "method": method_name,
            "task": "GlobalOpinionQA",
            "question_idx": idx,
            "question": question[:200],
            "response": resp[:500],
            "model_dist": model_dist.tolist(),
            "mean_jsd_vs_human": mean_jsd,
            "n_countries_compared": len(per_country_jsd),
            "trigger_rate": t_rate,
            "latency_s": latency,
            "steps": diag["steps"],
            "triggers": diag["triggers"],
            "mean_variance": diag["mean_variance"],
        })

    results_df = pd.DataFrame(results)

    # Aggregate country-level JSD
    country_mean_jsd = {c: float(np.mean(v)) for c, v in country_jsd_accum.items()}
    overall_jsd = float(np.mean(list(country_mean_jsd.values()))) if country_mean_jsd else np.nan

    summary = {
        "method": method_name,
        "overall_mean_jsd": overall_jsd,
        "n_questions": len(results),
        "n_countries": len(country_mean_jsd),
        "country_mean_jsd": country_mean_jsd,
        "mean_trigger_rate": float(results_df["trigger_rate"].mean()),
        "mean_latency_s": float(results_df["latency_s"].mean()),
        "compute_savings_pct": 100.0 - float(results_df["trigger_rate"].mean()),
    }

    print(f"[EVAL] GlobalOpinionQA Results:")
    print(f"  Overall Mean JSD:     {overall_jsd:.4f}")
    print(f"  Countries evaluated:  {len(country_mean_jsd)}")
    print(f"  Mean trigger rate:    {summary['mean_trigger_rate']:.1f}%")
    print(f"  Compute savings:      {summary['compute_savings_pct']:.1f}%")
    print(f"  Mean latency:         {summary['mean_latency_s']:.2f}s")

    return results_df, summary


# ============================================================================
# 6. EVALUATION: BLEnD MCQ (Cross-Linguistic Cultural Accuracy)
# ============================================================================
def _extract_choice_from_response(response: str) -> Optional[str]:
    """Extract answer choice (A/B/C/D) from generated text."""
    resp = response.strip()
    # Try JSON format first: {"answer_choice": "A"}
    try:
        parsed = json.loads(resp)
        if "answer_choice" in parsed:
            return parsed["answer_choice"].strip().upper()
    except (json.JSONDecodeError, AttributeError):
        pass
    # Try to find standalone letter at the beginning
    for ch in ["A", "B", "C", "D"]:
        if resp.upper().startswith(ch + ".") or resp.upper().startswith(ch + ")"):
            return ch
        if resp.upper().startswith(ch) and (len(resp) == 1 or not resp[1].isalpha()):
            return ch
    # Search in text
    for ch in ["A", "B", "C", "D"]:
        if f'"{ch}"' in resp or f"'{ch}'" in resp:
            return ch
    return None


def evaluate_blend_mcq(
    engine, df: pd.DataFrame, cfg: Phase2Config, method_name: str = "SWA_MPPI",
) -> Tuple[pd.DataFrame, Dict]:
    """
    Evaluate on BLEnD MCQ: measure cross-linguistic cultural accuracy.
    """
    print(f"\n[EVAL] BLEnD MCQ — {len(df)} questions, {df['country'].nunique()} countries")
    results = []

    for idx, row in tqdm(df.iterrows(), total=len(df), desc="BLEnD MCQ"):
        prompt_text = row.get("prompt", row.get("question", ""))
        country = row["country"]
        raw = row.get("answer_idx", row.get("answer", "A"))
        if isinstance(raw, (int, np.integer)) and 0 <= raw <= 3:
            correct_answer = "ABCD"[raw]
        else:
            correct_answer = str(raw).strip().upper()

        t0 = time.time()
        resp, t_rate, diag = engine.generate(prompt_text)
        latency = time.time() - t0

        if isinstance(diag, dict) and "samples" in diag:
            votes = []
            for s in diag.get("samples", []):
                ch = _extract_choice_from_response(s)
                if ch:
                    votes.append(ch)
            if votes:
                predicted = max(set(votes), key=votes.count)
            else:
                predicted = _extract_choice_from_response(resp)
        else:
            predicted = _extract_choice_from_response(resp)
        is_correct = (predicted == correct_answer) if predicted else False

        results.append({
            "method": method_name,
            "task": "BLEnD_MCQ",
            "question_id": row.get("MCQID", idx),
            "country": country,
            "prompt": prompt_text[:200],
            "response": resp[:300],
            "predicted": predicted,
            "correct": correct_answer,
            "is_correct": is_correct,
            "trigger_rate": t_rate,
            "latency_s": latency,
            "steps": diag["steps"],
            "triggers": diag["triggers"],
            "mean_variance": diag["mean_variance"],
        })

    results_df = pd.DataFrame(results)

    # Per-country accuracy
    country_acc = results_df.groupby("country")["is_correct"].mean().to_dict()
    overall_acc = float(results_df["is_correct"].mean())

    summary = {
        "method": method_name,
        "overall_accuracy": overall_acc,
        "n_questions": len(results),
        "n_countries": results_df["country"].nunique(),
        "country_accuracy": country_acc,
        "mean_trigger_rate": float(results_df["trigger_rate"].mean()),
        "mean_latency_s": float(results_df["latency_s"].mean()),
        "compute_savings_pct": 100.0 - float(results_df["trigger_rate"].mean()),
    }

    print(f"[EVAL] BLEnD MCQ Results:")
    print(f"  Overall Accuracy:     {overall_acc:.1%}")
    print(f"  Countries evaluated:  {results_df['country'].nunique()}")
    print(f"  Mean trigger rate:    {summary['mean_trigger_rate']:.1f}%")
    print(f"  Compute savings:      {summary['compute_savings_pct']:.1f}%")
    for c, acc in sorted(country_acc.items(), key=lambda x: -x[1]):
        print(f"    {c:20s}: {acc:.1%}")

    return results_df, summary


# ============================================================================
# 7B. METHOD COMPARISONS (paper-style figures/tables)
# ============================================================================
def plot_method_comparison(go_summaries: List[Dict], blend_summaries: List[Dict], output_dir: Path):
    """
    Fig P2-5: Compare methods across tasks (JSD, Acc, Latency, Savings).
    """
    if not go_summaries and not blend_summaries:
        return

    rows = []
    for s in go_summaries:
        rows.append({
            "method": s["method"],
            "task": "GlobalOpinionQA",
            "metric": "Mean JSD ↓",
            "value": s["overall_mean_jsd"],
        })
        rows.append({
            "method": s["method"],
            "task": "GlobalOpinionQA",
            "metric": "Latency (s) ↓",
            "value": s["mean_latency_s"],
        })
        rows.append({
            "method": s["method"],
            "task": "GlobalOpinionQA",
            "metric": "Savings (%) ↑",
            "value": s["compute_savings_pct"],
        })
    for s in blend_summaries:
        rows.append({
            "method": s["method"],
            "task": "BLEnD MCQ",
            "metric": "Accuracy ↑",
            "value": s["overall_accuracy"] * 100.0,
        })
        rows.append({
            "method": s["method"],
            "task": "BLEnD MCQ",
            "metric": "Latency (s) ↓",
            "value": s["mean_latency_s"],
        })
        rows.append({
            "method": s["method"],
            "task": "BLEnD MCQ",
            "metric": "Savings (%) ↑",
            "value": s["compute_savings_pct"],
        })

    df = pd.DataFrame(rows)
    if df.empty:
        return

    fig, ax = plt.subplots(figsize=(14, 6))
    sns.barplot(data=df, x="metric", y="value", hue="method", ax=ax)
    ax.set_title("Phase 2: Method Comparison (SWA-MPPI vs Baselines)", fontsize=13, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.legend(fontsize=9, title="Method")
    plt.tight_layout()
    path = output_dir / "fig_p2_5_method_comparison.png"
    plt.savefig(path, bbox_inches="tight")
    plt.close()
    print(f"[FIG P2-5] Saved → {path}")


def plot_globalopinion_jsd_by_method(go_results_all: pd.DataFrame, output_dir: Path):
    """
    Fig P2-6: Distribution of per-question mean JSD by method.
    """
    if go_results_all is None or go_results_all.empty or "method" not in go_results_all.columns:
        return
    df = go_results_all.copy()
    df = df.dropna(subset=["mean_jsd_vs_human"])
    if df.empty:
        return
    fig, ax = plt.subplots(figsize=(12, 5))
    sns.boxplot(data=df, x="method", y="mean_jsd_vs_human", ax=ax)
    ax.set_title("GlobalOpinionQA: Per-question Mean JSD by Method (↓ better)", fontsize=13, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel("Mean JSD vs Human")
    ax.tick_params(axis="x", rotation=20)
    plt.tight_layout()
    path = output_dir / "fig_p2_6_globalopinion_jsd_by_method.png"
    plt.savefig(path, bbox_inches="tight")
    plt.close()
    print(f"[FIG P2-6] Saved → {path}")


# ============================================================================
# 7. PUBLICATION-QUALITY VISUALIZATIONS
# ============================================================================
def plot_globalopinion_jsd(go_summary: Dict, output_dir: Path):
    """Fig P2-1: Per-country mean JSD for GlobalOpinionQA."""
    country_jsd = go_summary["country_mean_jsd"]
    if not country_jsd:
        print("[WARN] No country JSD data to plot.")
        return

    # Sort by JSD
    sorted_items = sorted(country_jsd.items(), key=lambda x: x[1])
    countries = [c for c, _ in sorted_items]
    jsds = [j for _, j in sorted_items]
    mean_jsd = go_summary["overall_mean_jsd"]

    fig, ax = plt.subplots(figsize=(12, max(6, len(countries) * 0.35)))
    colors = ["#2196F3" if j <= mean_jsd else "#FF9800" for j in jsds]
    bars = ax.barh(range(len(countries)), jsds, color=colors, edgecolor="white", height=0.7)
    ax.set_yticks(range(len(countries)))
    ax.set_yticklabels(countries, fontsize=9)
    ax.set_xlabel("Mean Jensen-Shannon Distance ↓", fontsize=12)
    ax.set_title("GlobalOpinionQA: SWA-MPPI vs Human Survey Distributions",
                 fontsize=13, fontweight="bold")
    ax.axvline(mean_jsd, color="#E53935", linestyle="--", linewidth=1.5,
               label=f"Global Mean JSD = {mean_jsd:.4f}")
    ax.invert_yaxis()
    ax.legend(fontsize=10)

    for i, (bar, val) in enumerate(zip(bars, jsds)):
        ax.text(val + 0.003, i, f"{val:.4f}", va="center", fontsize=8)

    path = output_dir / "fig_p2_1_globalopinion_jsd.pdf"
    plt.savefig(path, bbox_inches="tight")
    plt.savefig(str(path).replace(".pdf", ".png"))
    plt.close()
    print(f"[FIG P2-1] Saved → {path}")


def plot_blend_accuracy(blend_summary: Dict, output_dir: Path):
    """Fig P2-2: Per-country BLEnD MCQ accuracy."""
    country_acc = blend_summary["country_accuracy"]
    if not country_acc:
        print("[WARN] No country accuracy data to plot.")
        return

    sorted_items = sorted(country_acc.items(), key=lambda x: -x[1])
    countries = [c for c, _ in sorted_items]
    accs = [a * 100 for _, a in sorted_items]
    overall = blend_summary["overall_accuracy"] * 100

    fig, ax = plt.subplots(figsize=(10, max(5, len(countries) * 0.4)))
    colors = ["#4CAF50" if a >= overall else "#FF9800" for a in accs]
    bars = ax.barh(range(len(countries)), accs, color=colors, edgecolor="white", height=0.7)
    ax.set_yticks(range(len(countries)))
    ax.set_yticklabels(countries, fontsize=10)
    ax.set_xlabel("Accuracy (%)", fontsize=12)
    ax.set_title("BLEnD MCQ: Cross-Linguistic Cultural Accuracy (SWA-MPPI)",
                 fontsize=13, fontweight="bold")
    ax.axvline(overall, color="#E53935", linestyle="--", linewidth=1.5,
               label=f"Overall Accuracy = {overall:.1f}%")
    ax.axvline(25, color="#999999", linestyle=":", linewidth=1, alpha=0.5,
               label="Random Baseline (25%)")
    ax.invert_yaxis()
    ax.set_xlim(0, 100)
    ax.legend(fontsize=10, loc="lower right")

    for i, (bar, val) in enumerate(zip(bars, accs)):
        ax.text(val + 1, i, f"{val:.1f}%", va="center", fontsize=9)

    path = output_dir / "fig_p2_2_blend_accuracy.pdf"
    plt.savefig(path, bbox_inches="tight")
    plt.savefig(str(path).replace(".pdf", ".png"))
    plt.close()
    print(f"[FIG P2-2] Saved → {path}")


def plot_compute_savings(go_results: pd.DataFrame, blend_results: pd.DataFrame,
                          output_dir: Path):
    """Fig P2-3: MPPI compute savings distribution (variance trigger analysis)."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # (a) Trigger rate distribution
    ax1 = axes[0]
    all_trigger = pd.concat([
        go_results[["trigger_rate"]].assign(task="GlobalOpinionQA"),
        blend_results[["trigger_rate"]].assign(task="BLEnD MCQ"),
    ])
    savings = 100 - all_trigger["trigger_rate"]
    sns.histplot(savings, bins=20, color="#3498db", kde=True, ax=ax1, alpha=0.7)
    ax1.axvline(savings.mean(), color="#E53935", linestyle="--", linewidth=2,
                label=f"Mean Savings: {savings.mean():.1f}%")
    ax1.set_xlabel("Compute Savings (% MPPI bypassed)", fontsize=11)
    ax1.set_ylabel("Frequency", fontsize=11)
    ax1.set_title("(a) MPPI Bypass Rate Distribution", fontsize=12, fontweight="bold")
    ax1.legend(fontsize=10)

    # (b) Per-task comparison
    ax2 = axes[1]
    go_savings = 100 - go_results["trigger_rate"]
    bl_savings = 100 - blend_results["trigger_rate"]
    data = pd.DataFrame({
        "Savings (%)": pd.concat([go_savings, bl_savings]),
        "Task": (["GlobalOpinionQA"] * len(go_savings) + ["BLEnD MCQ"] * len(bl_savings)),
    })
    sns.boxplot(data=data, x="Task", y="Savings (%)", palette=["#2196F3", "#4CAF50"],
                ax=ax2, width=0.5)
    ax2.set_title("(b) Savings by Task", fontsize=12, fontweight="bold")
    ax2.set_ylabel("Compute Savings (%)", fontsize=11)

    # (c) Latency comparison
    ax3 = axes[2]
    lat_data = pd.DataFrame({
        "Latency (s)": pd.concat([go_results["latency_s"], blend_results["latency_s"]]),
        "Task": (["GlobalOpinionQA"] * len(go_results) + ["BLEnD MCQ"] * len(blend_results)),
    })
    sns.boxplot(data=lat_data, x="Task", y="Latency (s)", palette=["#2196F3", "#4CAF50"],
                ax=ax3, width=0.5)
    ax3.set_title("(c) Generation Latency", fontsize=12, fontweight="bold")
    ax3.set_ylabel("Latency (seconds)", fontsize=11)

    plt.tight_layout()
    path = output_dir / "fig_p2_3_compute_savings.pdf"
    plt.savefig(path, bbox_inches="tight")
    plt.savefig(str(path).replace(".pdf", ".png"))
    plt.close()
    print(f"[FIG P2-3] Saved → {path}")


def plot_phase2_summary_table(go_summary: Dict, blend_summary: Dict, output_dir: Path):
    """Fig P2-4: Summary results table."""
    rows = [
        ["GlobalOpinionQA", f"{go_summary['overall_mean_jsd']:.4f}",
         "—", f"{go_summary['n_countries']}", f"{go_summary['n_questions']}",
         f"{go_summary['mean_trigger_rate']:.1f}%",
         f"{go_summary['compute_savings_pct']:.1f}%",
         f"{go_summary['mean_latency_s']:.2f}s"],
        ["BLEnD MCQ", "—",
         f"{blend_summary['overall_accuracy']:.1%}",
         f"{blend_summary['n_countries']}", f"{blend_summary['n_questions']}",
         f"{blend_summary['mean_trigger_rate']:.1f}%",
         f"{blend_summary['compute_savings_pct']:.1f}%",
         f"{blend_summary['mean_latency_s']:.2f}s"],
    ]
    columns = ["Benchmark", "Mean JSD ↓", "Accuracy ↑", "Countries",
               "Questions", "Trigger Rate", "Savings", "Latency"]

    fig, ax = plt.subplots(figsize=(14, 2.5))
    ax.axis("off")
    table = ax.table(cellText=rows, colLabels=columns, cellLoc="center", loc="center")
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    table.scale(1, 2.0)

    for j in range(len(columns)):
        cell = table[0, j]
        cell.set_facecolor("#2196F3")
        cell.set_text_props(color="white", fontweight="bold")
    for i in range(1, len(rows) + 1):
        for j in range(len(columns)):
            table[i, j].set_facecolor("#F5F5F5" if i % 2 == 0 else "white")

    ax.set_title("Table 2: Phase 2 Open-Domain Evaluation Results (SWA-MPPI)",
                 fontsize=13, fontweight="bold", pad=20)

    path = output_dir / "fig_p2_4_summary_table.pdf"
    plt.savefig(path, bbox_inches="tight")
    plt.savefig(str(path).replace(".pdf", ".png"))
    plt.close()
    print(f"[FIG P2-4] Saved → {path}")

    # LaTeX table
    latex_path = output_dir / "table2_phase2_results.tex"
    with open(latex_path, "w") as f:
        f.write("\\begin{table}[t]\n\\centering\n")
        f.write("\\caption{Phase 2 Open-Domain Evaluation Results (SWA-MPPI)}\n")
        f.write("\\label{tab:phase2_results}\n\\small\n")
        f.write("\\begin{tabular}{l" + "c" * (len(columns) - 1) + "}\n")
        f.write("\\toprule\n")
        f.write(" & ".join(columns) + " \\\\\n\\midrule\n")
        for row in rows:
            f.write(" & ".join(row) + " \\\\\n")
        f.write("\\bottomrule\n\\end{tabular}\n\\end{table}\n")
    print(f"[TABLE] Saved LaTeX → {latex_path}")


# ============================================================================
# 8. MAIN EXECUTION PIPELINE
# ============================================================================
def main():
    cfg = parse_args()
    output_dir = Path(cfg.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # Resolve data directory (Kaggle: auto-detect from /kaggle/input if needed)
    if _is_kaggle():
        data_dir = _resolve_kaggle_data_dir(cfg.data_dir)
        if os.environ.get("SWA_MPPI_OUTPUT"):
            output_dir = Path(os.environ["SWA_MPPI_OUTPUT"])
            output_dir.mkdir(parents=True, exist_ok=True)
    else:
        data_dir = Path(cfg.data_dir)

    if not data_dir.exists():
        raise FileNotFoundError(
            f"Data directory not found: {data_dir}. "
            "On Kaggle, add the dataset containing llm_global_opinions_train.csv, "
            "SPB-2602_train.csv, and BLEnD_multiple-choice-questions_test.csv."
        )

    if _is_kaggle():
        print("[ENV] Running on Kaggle — data/output paths resolved for /kaggle/input and /kaggle/working")
    if not torch.cuda.is_available():
        print("[WARN] CUDA not available; generation will be slow or fail. On Kaggle, enable GPU (Settings → Accelerator).")

    print(f"{'='*70}")
    print(f"  SWA-MPPI Phase 2: Open-Domain Evaluation")
    print(f"  Model:    {cfg.model_name}")
    print(f"  Data:     {data_dir}")
    print(f"  Output:   {output_dir}")
    print(f"  Agents:   {cfg.num_agents}")
    print(f"  λ={cfg.lambda_coop}  α={cfg.alpha_kl}  K={cfg.K_union}  τ={cfg.tau_conflict}")
    print(f"{'='*70}\n")

    # ── 1. Load data ──
    print("[*] Loading datasets...")
    df_globalopinion = load_globalopinion_data(data_dir, cfg.num_eval_globalopinion)
    df_blend = load_blend_mcq_data(data_dir, cfg.num_eval_blend)
    personas = load_personas(data_dir, cfg.num_agents)

    # ── 2. Load model ──
    print("[*] Loading model...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=cfg.model_name,
        max_seq_length=cfg.max_seq_length,
        load_in_4bit=cfg.load_in_4bit,
    )
    FastLanguageModel.for_inference(model)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    tokenizer.padding_side = "left"
    print(f"[MODEL] VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

    # ── 3. Build engines (SWA-MPPI + baselines) ──
    engines = []
    engines.append(("SWA_MPPI", Phase2SWAEngine(cfg, model, tokenizer, personas, name="SWA_MPPI", enable_mppi=True, enable_variance_trigger=True)))
    engines.append(("SWA_NO_MPPI", Phase2SWAEngine(cfg, model, tokenizer, personas, name="SWA_NO_MPPI", enable_mppi=False, enable_variance_trigger=False)))
    engines.append(("SWA_ALWAYS_MPPI", Phase2SWAEngine(cfg, model, tokenizer, personas, name="SWA_ALWAYS_MPPI", enable_mppi=True, enable_variance_trigger=False)))
    engines.append(("BASE_GREEDY", BaseGenerationEngine(cfg, model, tokenizer, "BASE_GREEDY", do_sample=False)))
    engines.append(("BASE_TOPP_0.9_T0.7", BaseGenerationEngine(cfg, model, tokenizer, "BASE_TOPP_0.9_T0.7", do_sample=True, top_p=0.9, temperature=0.7)))
    engines.append(("BASE_TOPK_50_T0.8", BaseGenerationEngine(cfg, model, tokenizer, "BASE_TOPK_50_T0.8", do_sample=True, top_k=50, temperature=0.8)))
    engines.append(("SC_TOPP5", SelfConsistencyEngine(cfg, model, tokenizer, "SC_TOPP5", n_samples=5, top_p=0.9, temperature=0.7)))

    # ── 4. GlobalOpinionQA evaluation (all methods) ──
    print(f"\n{'='*70}")
    print("PHASE 2A: GlobalOpinionQA Evaluation (All Methods)")
    print(f"{'='*70}")
    all_go_results = []
    all_go_summaries = []
    for method_name, engine in engines:
        print(f"\n--- Method: {method_name} ---")
        go_results, go_summary = evaluate_globalopinion(engine, df_globalopinion, cfg, method_name=method_name)
        all_go_results.append(go_results)
        all_go_summaries.append(go_summary)
        go_results.to_csv(output_dir / f"phase2_globalopinion_results__{method_name}.csv", index=False)
    go_results_merged = pd.concat(all_go_results, ignore_index=True)
    go_results_merged.to_csv(output_dir / "phase2_globalopinion_results__ALL.csv", index=False)

    torch.cuda.empty_cache()
    gc.collect()

    # ── 5. BLEnD MCQ evaluation (all methods) ──
    print(f"\n{'='*70}")
    print("PHASE 2B: BLEnD MCQ Evaluation (All Methods)")
    print(f"{'='*70}")
    all_blend_results = []
    all_blend_summaries = []
    for method_name, engine in engines:
        print(f"\n--- Method: {method_name} ---")
        blend_results, blend_summary = evaluate_blend_mcq(engine, df_blend, cfg, method_name=method_name)
        all_blend_results.append(blend_results)
        all_blend_summaries.append(blend_summary)
        blend_results.to_csv(output_dir / f"phase2_blend_results__{method_name}.csv", index=False)
    blend_results_merged = pd.concat(all_blend_results, ignore_index=True)
    blend_results_merged.to_csv(output_dir / "phase2_blend_results__ALL.csv", index=False)

    torch.cuda.empty_cache()
    gc.collect()

    # ── 6. Save summaries ──
    all_summaries = {
        "globalopinion_by_method": all_go_summaries,
        "blend_by_method": all_blend_summaries,
        "config": {
            "model_name": cfg.model_name,
            "lambda_coop": cfg.lambda_coop,
            "alpha_kl": cfg.alpha_kl,
            "K_union": cfg.K_union,
            "tau_conflict": cfg.tau_conflict,
            "mppi_trajectories": cfg.mppi_trajectories,
            "num_agents": cfg.num_agents,
        },
    }
    with open(output_dir / "phase2_summaries.pkl", "wb") as f:
        pickle.dump(all_summaries, f)

    # ── 7. Generate all figures ──
    print(f"\n{'='*70}")
    print("GENERATING FIGURES")
    print(f"{'='*70}")
    # Use SWA-MPPI (first engine) as the primary method for legacy figures
    swa_go = next(s for s in all_go_summaries if s["method"] == "SWA_MPPI")
    swa_bl = next(s for s in all_blend_summaries if s["method"] == "SWA_MPPI")
    swa_go_df = next(df for df in all_go_results if df["method"].iloc[0] == "SWA_MPPI")
    swa_bl_df = next(df for df in all_blend_results if df["method"].iloc[0] == "SWA_MPPI")

    plot_globalopinion_jsd(swa_go, output_dir)
    plot_blend_accuracy(swa_bl, output_dir)
    plot_compute_savings(swa_go_df, swa_bl_df, output_dir)
    plot_phase2_summary_table(swa_go, swa_bl, output_dir)
    plot_method_comparison(all_go_summaries, all_blend_summaries, output_dir)
    plot_globalopinion_jsd_by_method(go_results_merged, output_dir)

    # ── 8. Final report ──
    print(f"\n{'='*70}")
    print("PHASE 2 FINAL REPORT")
    print(f"{'='*70}")
    for s in all_go_summaries:
        print(f"  [GlobalOpinionQA] {s['method']}: meanJSD={s['overall_mean_jsd']:.4f}  latency={s['mean_latency_s']:.2f}s  savings={s['compute_savings_pct']:.1f}%")
    for s in all_blend_summaries:
        print(f"  [BLEnD MCQ]       {s['method']}: acc={s['overall_accuracy']:.1%}  latency={s['mean_latency_s']:.2f}s  savings={s['compute_savings_pct']:.1f}%")

    # List all generated files
    print(f"\n{'='*70}")
    print(f"ALL OUTPUT FILES ({output_dir}):")
    print(f"{'='*70}")
    for f in sorted(output_dir.glob("*")):
        size_kb = f.stat().st_size / 1024
        print(f"  {f.name:45s} ({size_kb:.1f} KB)")

    print(f"\n[DONE] Phase 2 evaluation complete.")


if __name__ == "__main__":
    main()


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
[CONFIG] Fast mode (Kaggle): 8 GlobalOpinion, 16 BLEnD, max_new_tokens=64 (~20–40 min)
[ENV] Running on Kaggle — data/output paths resolved for /kaggle/input and /kaggle/working
  SWA-MPPI Phase 2: Open-Domain Evaluation
  Model:    unsloth/Meta-Llama-3.1-70B-Instruct-bnb-4bit
  Data:     /kaggle/input/datasets/minh2duy/swa-mppi-phase2-data
  Output:   /kaggle/working/SWA_MPPI_Phase2/results
  Agents:   5
  λ=0.7  α=0.05  K=50  τ=0.005

[*] Loading datasets...
[DATA] Loaded 8 GlobalOpinionQA questions
[DATA] Loaded 16 BLEnD MCQ questions (16 countries)
[DATA] Loaded 5 demographic personas from SPB-2602
[*] Loading model...
==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 1. Max memory: 79.179 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. 

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

...

[EVAL] BLEnD MCQ Results:
  Overall Accuracy:     87.5%
  Countries evaluated:  16
  Mean trigger rate:    0.0%
  Compute savings:      100.0%
    Azerbaijan          : 100.0%
    China               : 100.0%
    Ethiopia            : 100.0%
    Greece              : 100.0%
    Indonesia           : 100.0%
    Iran                : 100.0%
    Mexico              : 100.0%
    North_Korea         : 100.0%
    Northern_Nigeria    : 100.0%
    South_Korea         : 100.0%
    Spain               : 100.0%
    UK                  : 100.0%
    US                  : 100.0%
    West_Java           : 100.0%
    Algeria             : 0.0%
    Assam               : 0.0%

--- Method: BASE_TOPK_50_T0.8 ---

[EVAL] BLEnD MCQ — 16 questions, 16 countries


BLEnD MCQ:   0%|          | 0/16 [00:00<?, ?it/s]

[EVAL] BLEnD MCQ Results:
  Overall Accuracy:     93.8%
  Countries evaluated:  16
  Mean trigger rate:    0.0%
  Compute savings:      100.0%
    Assam               : 100.0%
    Azerbaijan          : 100.0%
    China               : 100.0%
    Ethiopia            : 100.0%
    Greece              : 100.0%
    Indonesia           : 100.0%
    Iran                : 100.0%
    Mexico              : 100.0%
    North_Korea         : 100.0%
    Northern_Nigeria    : 100.0%
    South_Korea         : 100.0%
    Spain               : 100.0%
    UK                  : 100.0%
    US                  : 100.0%
    West_Java           : 100.0%
    Algeria             : 0.0%

--- Method: SC_TOPP5 ---

[EVAL] BLEnD MCQ — 16 questions, 16 countries


BLEnD MCQ:   0%|          | 0/16 [00:00<?, ?it/s]

[EVAL] BLEnD MCQ Results:
  Overall Accuracy:     87.5%
  Countries evaluated:  16
  Mean trigger rate:    0.0%
  Compute savings:      100.0%
    Azerbaijan          : 100.0%
    China               : 100.0%
    Ethiopia            : 100.0%
    Greece              : 100.0%
    Indonesia           : 100.0%
    Iran                : 100.0%
    Mexico              : 100.0%
    North_Korea         : 100.0%
    Northern_Nigeria    : 100.0%
    South_Korea         : 100.0%
    Spain               : 100.0%
    UK                  : 100.0%
    US                  : 100.0%
    West_Java           : 100.0%
    Algeria             : 0.0%
    Assam               : 0.0%

GENERATING FIGURES
[FIG P2-1] Saved → /kaggle/working/SWA_MPPI_Phase2/results/fig_p2_1_globalopinion_jsd.pdf
[FIG P2-2] Saved → /kaggle/working/SWA_MPPI_Phase2/results/fig_p2_2_blend_accuracy.pdf
[FIG P2-3] Saved → /kaggle/working/SWA_MPPI_Phase2/results/fig_p2_3_compute_savings.pdf
[FIG P2-4] Saved → /kaggle/working/SWA_MPPI_Phas